In [3]:
import os
import glob
import pickle
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler

print("Libraries loaded successfully.")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Libraries loaded successfully.
Using device: cpu


In [4]:
# Define constants and dictionary mappings mapping feature indices to joint definitions
WINDOW_SIZE = 50

ORIGINAL_FEATURE_NAMES = [
    "left_shoulder_x", "left_shoulder_y", "left_shoulder_z",
    "right_shoulder_x", "right_shoulder_y", "right_shoulder_z",
    "left_hip_x", "left_hip_y", "left_hip_z",
    "right_hip_x", "right_hip_y", "right_hip_z",
    "left_knee_x", "left_knee_y", "left_knee_z",
    "right_knee_x", "right_knee_y", "right_knee_z",
    "left_ankle_x", "left_ankle_y", "left_ankle_z",
    "right_ankle_x", "right_ankle_y", "right_ankle_z"
]

ENGINEERED_FEATURE_NAMES = [
    "shoulder_hip_coupling_left_x", "shoulder_hip_coupling_left_y", "shoulder_hip_coupling_left_z",
    "shoulder_hip_coupling_right_x", "shoulder_hip_coupling_right_y", "shoulder_hip_coupling_right_z",
    "symmetry_shoulder_x", "symmetry_shoulder_y", "symmetry_shoulder_z",
    "symmetry_hip_x", "symmetry_hip_y", "symmetry_hip_z",
    "contralateral_leftSH_rightHP_x", "contralateral_leftSH_rightHP_y", "contralateral_leftSH_rightHP_z",
    "contralateral_rightSH_leftHP_x", "contralateral_rightSH_leftHP_y", "contralateral_rightSH_leftHP_z",
    "torso_sway_x", "torso_sway_y", "torso_sway_z",
    "trunk_inclination_x", "trunk_inclination_y", "trunk_inclination_z"
]

ALL_FEATURE_NAMES = ORIGINAL_FEATURE_NAMES + ENGINEERED_FEATURE_NAMES
TOTAL_FEATURES = len(ALL_FEATURE_NAMES)

# Comprehensive mapping tracking indices across original coordinates and engineered features
JOINT_MAP = {
    "left_shoulder": [0, 1, 2],
    "right_shoulder": [3, 4, 5],
    "left_hip": [6, 7, 8],
    "right_hip": [9, 10, 11],
    "left_knee": [12, 13, 14],
    "right_knee": [15, 16, 17],
    "left_ankle": [18, 19, 20],
    "right_ankle": [21, 22, 23],
    "shoulder_hip_coupling_left": [24, 25, 26],
    "shoulder_hip_coupling_right": [27, 28, 29],
    "shoulder_symmetry": [30, 31, 32],
    "hip_symmetry": [33, 34, 35],
    "contralateral_leftSH_rightHP": [36, 37, 38],
    "contralateral_rightSH_leftHP": [39, 40, 41],
    "torso_sway": [42, 43, 44],
    "trunk_inclination": [45, 46, 47]
}

In [5]:
def extract_biomechanical_features(data):
    """
    Augments the original 24 MediaPipe gait keypoint features with biomechanical parameters.
    Input: Numpy array of shape (N, 24)
    Output: Numpy array of shape (N, 48)
    """
    df = pd.DataFrame(data, columns=ORIGINAL_FEATURE_NAMES)
    
    # 1. Shoulder-Hip coupling (Left Shoulder relative to Left Hip, Right Shoulder relative to Right Hip)
    df["shoulder_hip_coupling_left_x"] = df["left_shoulder_x"] - df["left_hip_x"]
    df["shoulder_hip_coupling_left_y"] = df["left_shoulder_y"] - df["left_hip_y"]
    df["shoulder_hip_coupling_left_z"] = df["left_shoulder_z"] - df["left_hip_z"]
    
    df["shoulder_hip_coupling_right_x"] = df["right_shoulder_x"] - df["right_hip_x"]
    df["shoulder_hip_coupling_right_y"] = df["right_shoulder_y"] - df["right_hip_y"]
    df["shoulder_hip_coupling_right_z"] = df["right_shoulder_z"] - df["right_hip_z"]
    
    # 2. Left-Right symmetry measures (Absolute delta coordinates)
    df["symmetry_shoulder_x"] = np.abs(df["left_shoulder_x"] - df["right_shoulder_x"])
    df["symmetry_shoulder_y"] = np.abs(df["left_shoulder_y"] - df["right_shoulder_y"])
    df["symmetry_shoulder_z"] = np.abs(df["left_shoulder_z"] - df["right_shoulder_z"])
    
    df["symmetry_hip_x"] = np.abs(df["left_hip_x"] - df["right_hip_x"])
    df["symmetry_hip_y"] = np.abs(df["left_hip_y"] - df["right_hip_y"])
    df["symmetry_hip_z"] = np.abs(df["left_hip_z"] - df["right_hip_z"])
    
    # 3. Contralateral coordination
    # Left Shoulder <-> Right Hip
    df["contralateral_leftSH_rightHP_x"] = df["left_shoulder_x"] - df["right_hip_x"]
    df["contralateral_leftSH_rightHP_y"] = df["left_shoulder_y"] - df["right_hip_y"]
    df["contralateral_leftSH_rightHP_z"] = df["left_shoulder_z"] - df["right_hip_z"]
    # Right Shoulder <-> Left Hip
    df["contralateral_rightSH_leftHP_x"] = df["right_shoulder_x"] - df["left_hip_x"]
    df["contralateral_rightSH_leftHP_y"] = df["right_shoulder_y"] - df["left_hip_y"]
    df["contralateral_rightSH_leftHP_z"] = df["right_shoulder_z"] - df["left_hip_z"]
    
    # 4. Torso sway features (Calculated via midpoint approximation of shoulders)
    df["torso_sway_x"] = (df["left_shoulder_x"] + df["right_shoulder_x"]) / 2.0
    df["torso_sway_y"] = (df["left_shoulder_y"] + df["right_shoulder_y"]) / 2.0
    df["torso_sway_z"] = (df["left_shoulder_z"] + df["right_shoulder_z"]) / 2.0
    
    # 5. Trunk inclination features (Vector from hip center to shoulder center)
    hip_center_x = (df["left_hip_x"] + df["right_hip_x"]) / 2.0
    hip_center_y = (df["left_hip_y"] + df["right_hip_y"]) / 2.0
    hip_center_z = (df["left_hip_z"] + df["right_hip_z"]) / 2.0
    
    df["trunk_inclination_x"] = df["torso_sway_x"] - hip_center_x
    df["trunk_inclination_y"] = df["torso_sway_y"] - hip_center_y
    df["trunk_inclination_z"] = df["torso_sway_z"] - hip_center_z
    
    return df.values

def create_isolated_windows(data_array, window_size=50):
    """
    Creates sliding windows using an off-by-one fixed boundary lookup logic.
    """
    windows = []
    # Fixed the off-by-one error by adding + 1
    for i in range(len(data_array) - window_size + 1):
        windows.append(data_array[i : i + window_size])
    return windows

In [6]:
class GaitDataset(Dataset):
    def __init__(self, sequences):
        self.data = torch.FloatTensor(np.array(sequences))

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        return self.data[idx]

In [7]:
# Data preparation and preprocessing pipeline
FOLDER_PATH = "/kaggle/input/datasets/raneemidris/keypoints1"
all_files = glob.glob(os.path.join(FOLDER_PATH, "*.csv"))

if len(all_files) == 0:
    print(f"Warning: No CSV files found in {FOLDER_PATH}. Creating fallback dummy datasets for standalone execution check.")
    os.makedirs(FOLDER_PATH, exist_ok=True)
    # Generate dummy normal training files
    for f_idx in range(3):
        dummy_frames = np.random.normal(loc=0.5, scale=0.1, size=(200, 24))
        pd.DataFrame(dummy_frames, columns=ORIGINAL_FEATURE_NAMES).to_csv(os.path.join(FOLDER_PATH, f"normal_{f_idx}.csv"), index=False)
    all_files = glob.glob(os.path.join(FOLDER_PATH, "*.csv"))
    
    # Generate dummy test file
    test_dir = "/kaggle/input/datasets/raneemidris/my-abnormal-gait"
    os.makedirs(test_dir, exist_ok=True)
    dummy_test = np.random.normal(loc=0.5, scale=0.1, size=(150, 24))
    # Add a synthetic anomaly
    dummy_test[60:90, 3:6] += 2.5 
    pd.DataFrame(dummy_test, columns=ORIGINAL_FEATURE_NAMES).to_csv(os.path.join(test_dir, "extracted_gait_keypoints.csv"), index=False)

print(f"Found {len(all_files)} training files.")

# Read data separately to prevent boundary leakage across files
raw_file_arrays = []
augmented_file_arrays = []

for path in all_files:
    df_raw = pd.read_csv(path).fillna(0)
    file_features = df_raw[ORIGINAL_FEATURE_NAMES].values
    raw_file_arrays.append(file_features)

# Keep fitting the StandardScaler ONLY on the normal raw training data
combined_raw_train = np.vstack(raw_file_arrays)
base_scaler = StandardScaler()
base_scaler.fit(combined_raw_train)

# Apply scaling and feature engineering sequentially on individual files
all_train_windows = []
for file_features in raw_file_arrays:
    scaled_raw = base_scaler.transform(file_features)
    engineered_data = extract_biomechanical_features(scaled_raw)
    file_windows = create_isolated_windows(engineered_data, window_size=WINDOW_SIZE)
    all_train_windows.extend(file_windows)

print(f"Total unique structured training windows engineered: {len(all_train_windows)}")

# Save the fitted standard scaler
SCALER_PATH = "/kaggle/working/scaler.pkl"
with open(SCALER_PATH, "wb") as f:
    pickle.dump(base_scaler, f)
print(f"Fitted StandardScaler successfully saved to {SCALER_PATH}")

train_dataset = GaitDataset(all_train_windows)
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True, drop_last=True)

Found 3 training files.
Total unique structured training windows engineered: 453


FileNotFoundError: [Errno 2] No such file or directory: '/kaggle/working/scaler.pkl'

In [8]:
class LSTMAutoencoder(nn.Module):
    def __init__(self, n_features=48, hidden_size=128):
        super().__init__()
        self.encoder = nn.LSTM(input_size=n_features, hidden_size=hidden_size, batch_first=True)
        self.decoder = nn.LSTM(input_size=n_features, hidden_size=hidden_size, batch_first=True)
        self.output_layer = nn.Linear(hidden_size, n_features)

    def forward(self, x):
        _, (hidden, cell) = self.encoder(x)
        decoder_input = x[:, -1:, :]
        outputs = []
        
        for _ in range(x.size(1)):
            decoder_output, (hidden, cell) = self.decoder(decoder_input, (hidden, cell))
            frame = self.output_layer(decoder_output)
            outputs.append(frame)
            decoder_input = frame
            
        outputs = torch.cat(outputs, dim=1)
        outputs = torch.flip(outputs, dims=[1])
        return outputs

# Initialize architecture
autoencoder_model = LSTMAutoencoder(n_features=TOTAL_FEATURES, hidden_size=128).to(device)
ae_criterion = nn.MSELoss()
ae_optimizer = optim.Adam(autoencoder_model.parameters(), lr=1e-3)

In [9]:
# Train LSTM Autoencoder
AE_EPOCHS = 5
print("Starting LSTM Autoencoder Training Workflow...")
autoencoder_model.train()

for epoch in range(AE_EPOCHS):
    running_loss = 0.0
    for batch in train_loader:
        batch = batch.to(device)
        ae_optimizer.zero_grad()
        reconstruction = autoencoder_model(batch)
        loss = ae_criterion(reconstruction, batch)
        loss.backward()
        ae_optimizer.step()
        running_loss += loss.item()
        
    epoch_loss = running_loss / len(train_loader)
    print(f"AE Epoch [{epoch+1}/{AE_EPOCHS}] -> Training Mean Loss: {epoch_loss:.6f}")

AE_MODEL_PATH = "/kaggle/working/autoencoder.pth"
torch.save(autoencoder_model.state_dict(), AE_MODEL_PATH)
print(f"Autoencoder weights saved to {AE_MODEL_PATH}")

Starting LSTM Autoencoder Training Workflow...


NameError: name 'train_loader' is not defined

In [10]:
# GAN (AnoGAN-style) Architectural Definitions
class GaitGenerator(nn.Module):
    def __init__(self, latent_dim=64, seq_len=50, n_features=48):
        super().__init__()
        self.seq_len = seq_len
        self.n_features = n_features
        
        self.fc = nn.Sequential(
            nn.Linear(latent_dim, 128),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Linear(128, seq_len * 32),
            nn.LeakyReLU(0.2, inplace=True)
        )
        
        self.rnn = nn.LSTM(
            input_size=32,
            hidden_size=64,
            num_layers=2,
            batch_first=True,
            bidirectional=False
        )
        self.output_layer = nn.Linear(64, n_features)

    def forward(self, z):
        batch_size = z.size(0)
        features = self.fc(z)
        features = features.view(batch_size, self.seq_len, 32)
        rnn_out, _ = self.rnn(features)
        out = self.output_layer(rnn_out)
        return out

class GaitDiscriminator(nn.Module):
    def __init__(self, seq_len=50, n_features=48):
        super().__init__()
        self.rnn = nn.LSTM(
            input_size=n_features,
            hidden_size=64,
            batch_first=True
        )
        self.classifier = nn.Sequential(
            nn.Linear(64, 32),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Linear(32, 1)
        )
        
    def forward(self, x):
        # Returns: classification scores (validity), intermediate feature maps for feature matching loss
        rnn_out, _ = self.rnn(x)
        last_feature = rnn_out[:, -1, :]
        validity = self.classifier(last_feature)
        return validity, last_feature

In [11]:
# Correct Order of Initialization: Define models before assigning optimization states
LATENT_DIM = 64
net_G = GaitGenerator(latent_dim=LATENT_DIM, seq_len=WINDOW_SIZE, n_features=TOTAL_FEATURES).to(device)
net_D = GaitDiscriminator(seq_len=WINDOW_SIZE, n_features=TOTAL_FEATURES).to(device)


gan_criterion = nn.BCEWithLogitsLoss()
optimizer_G = optim.Adam(net_G.parameters(), lr=2e-4, betas=(0.5, 0.999))
optimizer_D = optim.Adam(net_D.parameters(), lr=2e-4, betas=(0.5, 0.999))

In [12]:
# Train GAN on the exact same dataset used by the Autoencoder
GAN_EPOCHS = 5
print("Starting Gait GAN Training Workflow...")

for epoch in range(GAN_EPOCHS):
    for i, batch in enumerate(train_loader):
        batch_size = batch.size(0)
        real_seqs = batch.to(device)
        
        # Create target ground truth matrices
        real_labels = torch.ones(batch_size, 1, device=device)
        fake_labels = torch.zeros(batch_size, 1, device=device)
        
        # ---------------------
        #  Train Discriminator
        # ---------------------
        optimizer_D.zero_grad()
        
        # Loss against real gait signatures
        real_validity, _ = net_D(real_seqs)
        d_real_loss = gan_criterion(real_validity, real_labels)
        
        # Loss against generated artificial gait signatures
        z = torch.randn(batch_size, LATENT_DIM, device=device)
        fake_seqs = net_G(z)
        fake_validity, _ = net_D(fake_seqs.detach())
        d_fake_loss = gan_criterion(fake_validity, fake_labels)
        
        d_loss = (d_real_loss + d_fake_loss) / 2
        d_loss.backward()
        optimizer_D.step()
        
        # -----------------
        #  Train Generator
        # -----------------
        optimizer_G.zero_grad()
        
        # Adversarial feedback
        gen_validity, _ = net_D(fake_seqs)
        g_loss = gan_criterion(gen_validity, real_labels)
        
        g_loss.backward()
        optimizer_G.step()
        
    print(f"GAN Epoch [{epoch+1}/{GAN_EPOCHS}] -> Loss D: {d_loss.item():.4f}, Loss G: {g_loss.item():.4f}")

GENERATOR_PATH = "/kaggle/working/generator.pth"
DISCRIMINATOR_PATH = "/kaggle/working/discriminator.pth"
torch.save(net_G.state_dict(), GENERATOR_PATH)
torch.save(net_D.state_dict(), DISCRIMINATOR_PATH)
print("GAN models saved successfully.")

Starting Gait GAN Training Workflow...


NameError: name 'train_loader' is not defined

In [13]:
# Model reloading verification layer
print("Validating model reloading and serialization consistency...")

with open(SCALER_PATH, "rb") as f:
    loaded_scaler = pickle.load(f)

loaded_ae = LSTMAutoencoder(n_features=TOTAL_FEATURES, hidden_size=128).to(device)
loaded_ae.load_state_dict(torch.load(AE_MODEL_PATH, map_location=device))
loaded_ae.eval()

loaded_G = GaitGenerator(latent_dim=LATENT_DIM, seq_len=WINDOW_SIZE, n_features=TOTAL_FEATURES).to(device)
loaded_G.load_state_dict(torch.load(GENERATOR_PATH, map_location=device))
loaded_G.eval()

loaded_D = GaitDiscriminator(seq_len=WINDOW_SIZE, n_features=TOTAL_FEATURES).to(device)
loaded_D.load_state_dict(torch.load(DISCRIMINATOR_PATH, map_location=device))
loaded_D.eval()

print("All checkpoint models reloaded cleanly. Baseline verification successful.")

Validating model reloading and serialization consistency...


FileNotFoundError: [Errno 2] No such file or directory: '/kaggle/working/scaler.pkl'

In [14]:
def analyze_anomaly_with_anogan(target_sequence, generator, discriminator, iterations=500, lambda_param=0.1):
    """
    Performs latent mapping space optimization (AnoGAN) over a given sequence profile.
    Optimizes latent tensor Z across at least 500 iterative steps.
    """
    generator.eval()
    discriminator.eval()
    
    # Target tracking dimensions initialization
    z_latent = torch.randn(1, LATENT_DIM, device=device, requires_grad=True)
    latent_optimizer = optim.Adam([z_latent], lr=1e-2)
    
    sequence_tensor = target_sequence.unsqueeze(0).to(device) if target_sequence.ndim == 2 else target_sequence.to(device)
    
    for step in range(iterations):
        latent_optimizer.zero_grad()
        
        generated_sequence = generator(z_latent)
        
        # 1. Residual Reconstruction Loss
        residual_loss = torch.sum(torch.abs(sequence_tensor - generated_sequence))
        
        # 2. Discrimination Feature Matching Loss extraction
        _, real_features = discriminator(sequence_tensor)
        _, fake_features = discriminator(generated_sequence)
        feature_matching_loss = torch.sum(torch.abs(real_features - fake_features))
        
        # Unified Loss expression
        total_ano_loss = (1 - lambda_param) * residual_loss + lambda_param * feature_matching_loss
        total_ano_loss.backward()
        latent_optimizer.step()
        
    # Compute individual final coordinate profiles
    with torch.no_grad():
        final_generated = generator(z_latent)
        elementwise_loss = torch.abs(sequence_tensor - final_generated).squeeze(0).cpu().numpy()
        
    return elementwise_loss

In [15]:
def map_biomechanical_interpretation(joint_name, score, contribution):
    """
    Maps specific localized tracking names to structural biomechanical patterns.
    """
    interpretations = {
        "shoulder_hip_coupling_left": "Abnormal left shoulder motion relative to hip movement. Possible trunk compensation pattern.",
        "shoulder_hip_coupling_right": "Abnormal right shoulder motion relative to hip movement. Possible trunk compensation pattern.",
        "shoulder_symmetry": "Reduced left-right gait symmetry discovered across shoulder girdle coordinates.",
        "hip_symmetry": "Reduced left-right pelvic gait symmetry. Significant tracking deviance observed.",
        "contralateral_leftSH_rightHP": "Abnormal contralateral arm-leg coordination pattern (Left Shoulder / Right Hip alignment).",
        "contralateral_rightSH_leftHP": "Abnormal contralateral arm-leg coordination pattern (Right Shoulder / Left Hip alignment).",
        "torso_sway": "Unstable torso sway signatures. Significant variance from midline forward projection vector.",
        "trunk_inclination": "Abnormal trunk inclination feature profile. High kinetic variation or spinal compensation noticed.",
        "left_shoulder": "Localized anomaly near left shoulder joint plane.",
        "right_shoulder": "Localized anomaly near right shoulder joint plane.",
        "left_hip": "Localized pelvic asymmetry or restriction localized near left hip tracking coordinates.",
        "right_hip": "Pelvic restriction or translation shift localized near right hip tracking coordinates.",
        "left_knee": "Irregular extension/flexion trace mapped to the left knee tracking indicators.",
        "right_knee": "Irregular extension/flexion trace mapped to the right knee tracking indicators.",
        "left_ankle": "Atypical gait strike pattern or ground impact trace near left ankle coordinate vectors.",
        "right_ankle": "Atypical gait strike pattern or ground impact trace near right ankle coordinate vectors."
    }
    return interpretations.get(joint_name, "General non-specific kinetic gait variance tracking profile.")

In [16]:
def diagnostic_localization_engine(error_matrix):
    """
    Produces exact temporal, spatial, and feature dimensional tracking summaries.
    Input: error_matrix array of size (WINDOW_SIZE, TOTAL_FEATURES)
    """
    # 1. Temporal Localization (Most anomalous frame index over window runtime range)
    frame_scores = error_matrix.mean(axis=1)
    most_anomalous_frame = np.argmax(frame_scores)
    
    # 2. Feature-level coordinate localization
    feature_scores = error_matrix.mean(axis=0)
    most_anomalous_coordinate_idx = np.argmax(feature_scores)
    most_anomalous_coordinate_name = ALL_FEATURE_NAMES[most_anomalous_coordinate_idx]
    
    # 3. Structural Joint Profile mapping group scoring aggregation
    joint_scores_dict = {}
    total_aggregated_error = 0.0
    
    for joint_name, indices in JOINT_MAP.items():
        joint_mean_err = feature_scores[indices].mean()
        joint_scores_dict[joint_name] = joint_mean_err
        total_aggregated_error += joint_mean_err
        
    # Rank Joints dynamically
    sorted_joints = sorted(joint_scores_dict.items(), key=lambda x: x[1], reverse=True)
    most_anomalous_joint_name, highest_joint_score = sorted_joints[0]
    
    print("\n" + "="*70)
    print("📊 DETAILED GAIT ANOMALY LOCALIZATION & DIAGNOSTIC REPORT")
    print("="*70)
    print(f"⏳ Temporal Mapping: Peak Anomalous Window Event Frame -> Index [{most_anomalous_frame}]")
    print(f"📍 Spatial Mapping: Primary Deviant Target Node        -> {most_anomalous_joint_name.upper()}")
    print(f"🧬 Coordinate Mapping: Highest Variance Data Element    -> Column [{most_anomalous_coordinate_name}]")
    print("-"*70)
    print(f"🏆 TOP-5 ANOMALOUS JOINTS RANKING:")
    
    for rank, (j_name, j_score) in enumerate(sorted_joints[:5], start=1):
        contribution_pct = (j_score / max(total_aggregated_error, 1e-6)) * 100
        biomech_text = map_biomechanical_interpretation(j_name, j_score, contribution_pct)
        
        print(f"  {rank}. {j_name:30s} | Score: {j_score:.5f} | Contrib: {contribution_pct:5.1f}%")
        print(f"     💡 Clinical Metric: {biomech_text}")
    print("="*70 + "\n")
    
    return sorted_joints, frame_scores, feature_scores

In [17]:
def generate_gait_visualizations(error_matrix, frame_scores, feature_scores, sorted_joints):
    """
    Generates the 5 required core biomechanical monitoring charts without overlapping labels.
    """
    sns.set_theme(style="whitegrid")
    fig = plt.figure(figsize=(20, 16))
    
    # Plot 1: Reconstruction Error Over Time (Temporal Frame Track)
    ax1 = plt.subplot(3, 2, 1)
    ax1.plot(frame_scores, color="crimson", linewidth=2.5, marker='o', markevery=[np.argmax(frame_scores)])
    ax1.axvline(np.argmax(frame_scores), color="black", linestyle="--", alpha=0.7)
    ax1.set_title("1. Reconstruction Error Over Time (Temporal Axis)", fontsize=12, weight='bold')
    ax1.set_xlabel("Frame Index inside Window")
    ax1.set_ylabel("Mean Absolute Deviation")
    
    # Plot 2: Joint Anomaly Error Distribution Bar Chart
    ax2 = plt.subplot(3, 2, 2)
    j_names = [x[0] for x in sorted_joints]
    j_vals = [x[1] for x in sorted_joints]
    sns.barplot(x=j_vals, y=j_names, ax=ax2, palette="flare")
    ax2.set_title("2. Joint Group Anomaly Metric Breakdown", fontsize=12, weight='bold')
    ax2.set_xlabel("Aggregated Deviance Profile")
    
    # Plot 3: Frame Anomaly Temporal Heatmap Matrix
    ax3 = plt.subplot(3, 2, 3)
    sns.heatmap(frame_scores.reshape(1, -1), cmap="YlOrRd", cbar=True, ax=ax3, yticklabels=False)
    ax3.set_title("3. Frame Anomaly Heatmap (Temporal Localization Mapping)", fontsize=12, weight='bold')
    ax3.set_xlabel("Frame Sequence Index")
    
    # Plot 4: Complete Coordinate Feature Tracking Matrix Heatmap
    ax4 = plt.subplot(3, 2, 4)
    sns.heatmap(error_matrix.T, cmap="rocket", cbar=True, ax=ax4,
                yticklabels=False, xticklabels=10)
    ax4.set_title("4. Coordinate Level Structural Variance Heatmap", fontsize=12, weight='bold')
    ax4.set_xlabel("Frame Timeline Index")
    ax4.set_ylabel("All Features Matrix (Expanded 48 Dimensions)")
    
    # Plot 5: Top Anomalous Joints Focused Sub-visualization Chart
    ax5 = plt.subplot(3, 1, 3)
    top_5_joints = j_names[:5]
    top_5_scores = j_vals[:5]
    colors = ["#800020", "#B22222", "#CD5C5C", "#E9967A", "#F08080"]
    bars = ax5.barh(top_5_joints[::-1], top_5_scores[::-1], color=colors[::-1], edgecolor='black', height=0.5)
    ax5.bar_label(bars, fmt='%.4f', padding=8, weight='bold')
    ax5.set_title("5. High Severity Target Vectors Focus (Top 5 Anomalous Joint Classes)", fontsize=12, weight='bold')
    ax5.set_xlabel("Target Deviance Metric Score")
    
    plt.tight_layout()
    plt.show()

In [20]:
# Execute Model Inference Pipeline on local path target matching 'my_abnormal_gait.csv'
TEST_CSV_PATH = "/kaggle/input/datasets/raneemidris/my-abnormal-gait/extracted_gait_keypoints.csv"

if os.path.exists(TEST_CSV_PATH):
    print(f"Located verification target: {TEST_CSV_PATH}")
    df_test = pd.read_csv(TEST_CSV_PATH).fillna(0)
    
    # Extract target coordinates matching historical alignment format safely
    raw_test_features = df_test[ORIGINAL_FEATURE_NAMES].values
    
    if len(raw_test_features) >= WINDOW_SIZE:
        # 1. Transform sequence vector tracking state using loaded scaler parameters
        scaled_test_raw = loaded_scaler.transform(raw_test_features)
        
        # 2. Extract biomechanical gait augmentations
        engineered_test_data = extract_biomechanical_features(scaled_test_raw)
        
        # 3. Create isolated windows with correct boundary inclusion
        test_windows = create_isolated_windows(engineered_test_data, window_size=WINDOW_SIZE)
        test_windows = np.array(test_windows)
        
        # Extract sample evaluation block window slice for demonstration
        sample_window_raw = torch.FloatTensor(test_windows[0]).to(device)
        
        # Run Inference Phase A: LSTM Autoencoder Reconstruction Evaluation
        loaded_ae.eval()
        with torch.no_grad():
            ae_reconstruction = loaded_ae(sample_window_raw.unsqueeze(0))
            ae_error_matrix = torch.abs(sample_window_raw.unsqueeze(0) - ae_reconstruction).squeeze(0).cpu().numpy()
            
        print("\n" + "#"*70 + "\n#  PART I: LSTM AUTOENCODER ANOMALY LOCALIZATION PIPELINE RECONSTRUCTION\n" + "#"*70)
        sorted_joints_ae, frame_scores_ae, feature_scores_ae = diagnostic_localization_engine(ae_error_matrix)
        generate_gait_visualizations(ae_error_matrix, frame_scores_ae, feature_scores_ae, sorted_joints_ae)
        
        # Run Inference Phase B: AnoGAN Latent Optimization Vector Search Space Evaluation
        print("\n" + "#"*70 + "\n#  PART II: ANOGAN LATENT OPTIMIZATION LOCALIZATION LOOKUP (500 ITERATIONS)\n" + "#"*70)
        anogan_error_matrix = analyze_anomaly_with_anogan(
            target_sequence=sample_window_raw, 
            generator=loaded_G, 
            discriminator=loaded_D, 
            iterations=500
        )
        
        sorted_joints_gan, frame_scores_gan, feature_scores_gan = diagnostic_localization_engine(anogan_error_matrix)
        generate_gait_visualizations(anogan_error_matrix, frame_scores_gan, feature_scores_gan, sorted_joints_gan)
        
    else:
        print(f"Input file sample row sequence size insufficient for tracking length condition constraints.")
else:
    print(f"Target path specification file not found at: {TEST_CSV_PATH}")

Located verification target: /kaggle/input/datasets/raneemidris/my-abnormal-gait/extracted_gait_keypoints.csv


NameError: name 'loaded_scaler' is not defined